In [ ]:
#import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date


sys.path.insert(0, '/projects/old_shared/fire_weather_vis/base-fwi-vis/')
import fwiVis.fwiVis as fv

In [ ]:
path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_plus_area_fix_09042024.csv" # Fires with merge fix + chaining fires fix + fires alseep fix
fires = fv.prep_fire_files(path, crs = "4326")

fires = fires.to_crs(4326)

In [ ]:
# Read in the fire area from the official perimeters

# import os
# import zipfile

# path = '/projects/old_shared/fire_weather_vis/NBAC/'

# os.chdir(path)

# for file in os.listdir('.'):
#     with zipfile.ZipFile(file, 'r') as zip_ref:
#         zip_ref.extractall('.')


nbac_only = gpd.read_file("/projects/old_shared/fire_weather_vis/NBAC/nbac_2023_20240530.shp")

nbac_only = nbac_only[nbac_only.ADMIN_AREA == 'QC']
nbac_only  = nbac_only [(nbac_only.ADJ_HA >= 500) |(nbac_only.POLY_HA >= 500)]

In [ ]:
fires = fires.to_crs(nbac_only.crs)
fires = fires[['fireID', 't', 'geometry','n_pixels',
       'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
       'meanFRP', 'lon_centroid',
       'lat_centroid', 'composit_ids', 'rows_edited']]


def get_max_area(df):
    df["unitless_area"] = df.geometry.area
    df = df[df.unitless_area == df.unitless_area.max()]
    return( df[df.t == df.t.max()])

fires = fires.groupby("fireID").apply(get_max_area).reset_index(drop = True)


In [ ]:
nbac = nbac_only.sjoin(fires)

In [ ]:

nbac = nbac[(nbac.HS_SDATE < "2023-09-15" ) |  (nbac.AG_SDATE < "2023-09-15")]


In [ ]:
print(len(nbac.NFIREID.unique()))

print(len(nbac_only.NFIREID.unique())) 
print(len(fires.fireID.unique()))

In [ ]:
## Fires not in nbac

feds_only = fires[~fires.fireID.isin(nbac.fireID)]
print(len(feds_only.fireID.unique()))
print(feds_only.groupby("fireID").farea.max().mean())
feds_only.explore()



## Small, short (1-4 time periods), and possibly tied to static sources and/or false positives. (2 Water bodies,  1 airport). 

In [ ]:
nbac_o = nbac_only[~nbac_only.NFIREID.isin(nbac.NFIREID)]
print(len(nbac_o.NFIREID.unique()))
print(nbac_o.groupby("NFIREID").ADJ_HA.max().mean())
nbac_o.explore() 

## 2 started late enough that it likely wasn't big enough for out analysis. Other 3  all small

In [ ]:
### Look at the fires we agree on. 

seq = range(0, round(nbac.farea.max()), round((nbac.farea.max()/ len(nbac.farea))))

plt.scatter(nbac.ADJ_HA/100, nbac.farea)
plt.ylabel("NBAC Fire Area km^2")
plt.xlabel("FEDS Fire Area km^2")

plt.plot(seq, seq)

In [ ]:
### Check the outliers

nbac["area_source_diff"] = (nbac.ADJ_HA/100) - (nbac.farea)
plus_10_percent = (nbac.ADJ_HA/100) >= (nbac.farea * 1.1)
minus_10_percent = (nbac.ADJ_HA/100) <= (nbac.farea * 0.9)



bigger_ids = nbac[plus_10_percent].NFIREID.unique() ## 19 larger


nbac = nbac.sort_values(by = "area_source_diff")
nbac.groupby("NFIREID").area_source_diff.max()

In [ ]:
?range